**t-SNE Projection of Image Data**

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns

X_flat = X_train.reshape(X_train.shape[0], -1)
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_flat)

sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_train, palette="coolwarm")
plt.title("t-SNE Projection of Image Data")
plt.show()

**Building the CNN Model**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Load dataset
images = np.load('data/images.npy')
labels = np.load('data/labels.npy')

# Normalize images to [0,1] range
images = images.astype('float32') / 255.0
images = np.expand_dims(images, axis=-1)  # Add channel dimension

# Train-Validation-Test Split with Shuffle and Stratify
X_train, X_temp, y_train, y_temp = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp, shuffle=True)

# Verify Data Splitting 
print("Train Set Size:", X_train.shape[0])
print("Validation Set Size:", X_val.shape[0])
print("Test Set Size:", X_test.shape[0])
print("Unique samples in Train:", len(set([tuple(x.flatten()) for x in X_train])))
print("Unique samples in Validation:", len(set([tuple(x.flatten()) for x in X_val])))
print("Unique samples in Test:", len(set([tuple(x.flatten()) for x in X_test])))

# CNN Model
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(16, 16, 1)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compile Model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Training Model with Early Stopping
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val), 
    epochs=50, batch_size=32, callbacks=[early_stopping]
)

# Plot Accuracy and Loss
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='Train Accuracy')
ax[0].plot(history.history['val_accuracy'], label='Val Accuracy')
ax[0].set_title('Accuracy')
ax[0].legend()
ax[1].plot(history.history['loss'], label='Train Loss')
ax[1].plot(history.history['val_loss'], label='Val Loss')
ax[1].set_title('Loss')
ax[1].legend()
plt.show()

# Sample Prediction
sample_image = X_test[0]
prediction = model.predict(sample_image.reshape(1, 16, 16, 1))
print("Predicted Label:", 1 if prediction > 0.5 else 0)


In [ ]:
def plot_predictions(model, X_test, y_test, num_samples=5):
    plt.figure(figsize=(10, 5))
    for i in range(num_samples):
        ax = plt.subplot(1, num_samples, i + 1)
        sample_image = X_test[i]
        prediction = model.predict(sample_image.reshape(1, 16, 16, 1))
        predicted_label = 1 if prediction > 0.5 else 0
        actual_label = y_test[i]
        
        plt.imshow(sample_image.squeeze(), cmap='gray')
        plt.title(f"Pred: {predicted_label}\nActual: {actual_label}")
        plt.axis("off")
    plt.show()

# Plotting predictions
plot_predictions(model, X_test, y_test)

In [ ]:
# Plotting Confusion Matrix
y_pred = model.predict(X_test)
y_pred_labels = (y_pred > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred_labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()